# Part 4 — GenAI: turn call-centre notes into structured reasons, then recommend (22 marks)

## Task 4.1 — One validated call

Get a free key in Google AI Studio and install the library in Colab. Write a function analyse_note(note_text) that sends one note to the Gemini Flash model used in class, with a system instruction, a prompt built with the CRAFT parts (Context, Role, Action, Tone, Format), temperature=0, response_mime_type set to JSON, and a JSON response schema with four fields: root_cause (a plain string — do not make it an enum; the allowed values go in your prompt and are enforced afterwards by the check in Task 4.2), confidence (0–1), summary (at most 25 words), recommended_action. Set the thinking level to minimal as shown in Session 22.1, and give max_output_tokens plenty of room (1000 or more): on some models hidden reasoning tokens count against the limit and can cut the JSON off, which makes json.loads fail. Run it on note N001 and print the parsed dictionary. Shape of the call (follow your Session 22.1 notebook if any detail differs; the key is typed at run time):

In [ ]:
import os
import json
import pandas as pd
from google import genai
from google.genai import types

SCHEMA = {   # same idea as the name / age / city example in class: properties, their types, and required
    "type": "object",
    "properties": {"root_cause": {"type": "string"}, "confidence": {"type": "number"},
                   "summary": {"type": "string"}, "recommended_action": {"type": "string"}},
    "required": ["root_cause", "confidence", "summary", "recommended_action"]}

API_KEY = input("Paste your key: ")

client = genai.Client(api_key=API_KEY)

MODEL_NAME = "gemini-3.5-flash"

def analyse_note(note_text, call_count=0):
  prompt = f"""
  You are a medical consultation followup note analysis system.

  Analyze the following patient's followup note.

  Extract:
  1. The root cause of the issue
  2. The confidence: value between 0 and 1
  3. A concise summary in not more than 25 words
  4. The recommended action for the followup

  Do not invent information that is not present in the note.

  Followup Note:
  {note_text}
  """

  # Define model behavior
  SYSTEM_TEXT="""
  You are a clinic reception assitant.

  Extract information accurately from patient followup notes.

  Allowed root_cause values are forgot_or_no_reminder, distance_or_transport,
  cost_or_fee, illness_or_emergency, work_or_schedule, other.

  Rules:
  - Do not invent information.
  - Only use information present in the notes.
  - Preserve names accurately.
  - Extract every explicit action item.
  - Return only the requested structured information.
  """

  response = client.models.generate_content(
      model=MODEL_NAME,                      # the Flash model from Session 22.1
      contents=prompt,
      config=types.GenerateContentConfig(system_instruction=SYSTEM_TEXT, temperature=0,
          max_output_tokens=1000, response_mime_type="application/json", response_schema=SCHEMA,
          thinking_config=types.ThinkingConfig(thinking_level="minimal")))   # as in Session 22.1; if your model rejects it, delete this line

  call_count = call_count + 1

  data = json.loads(response.text)

  return data, call_count

note = "Patient said she never got any reminder message and completely forgot about the visit. Wants to rebook for next week."

N001_analysis_resp, calls_made_for_4_1 = analyse_note(note)

print(f"\nResponse from the model is: {N001_analysis_resp}")



## Task 4.2 — Validate and process the 10 notes

(a) Define a Pydantic model FollowupInsight(BaseModel) with root_cause: str = Field(min_length=3), confidence: float = Field(ge=0, le=1), summary: str = Field(min_length=5), recommended_action: str = Field(min_length=5), and write a check that root_cause is in the allowed list.

(b) Read followup_notes.csv with pandas and loop over the first 10 rows. Input validation: pandas reads a blank note as NaN (a float, which has no .strip()), so test pd.isna(note_text) first, then str(note_text).strip() == "". If the note is missing, empty or only spaces, do not call the API — record root_cause="other", confidence=0.0, summary="No note recorded", recommended_action="Re-call patient" and validation_status="skipped_blank". Otherwise call analyse_note, validate the result with the Pydantic model (as in class: pass the dictionary into the class, FollowupInsight(**data)) and the allowed-list check, and set validation_status to "valid" or "invalid" (never let one bad reply stop the loop: use try/except).

(c) Pause between calls (import time then time.sleep(13)) to respect the free-tier rate limit.

(d) Save notes_analysed.csv with columns note_id, appointment_id, root_cause, confidence, summary, recommended_action, validation_status, raw_response. Keep every reply, including invalid ones, and never repair it: put the model's reply exactly as returned (the JSON text) in raw_response, copy whichever fields it did contain, and leave a field empty if the reply lacked it (for a skipped blank note, raw_response is empty).

In [ ]:
from pydantic import BaseModel, Field, ValidationError

# 4.2a: Create a Pydantic BaseModel for the insight output and validate the
# response from GenAI based on this class

class FollowupInsight(BaseModel):
  root_cause: str = Field(min_length=3)
  confidence: float = Field(ge=0, le=1)
  summary: str = Field(min_length=5)
  recommended_action: str = Field(min_length=5)

def validateResp(data):

  try:

    # insight = FollowupInsight.model_validate(data)
    insight = FollowupInsight(**data)

    print("Validation Successfull")
    print(f"Insight is: {insight}")
    print(f"Type of object: {type(insight)}")

    return True

  except ValidationError as ve:

    print("Data validation failed with the error:")
    print(ve)
    return False

validateResp(N001_analysis_resp)

In [ ]:
import time

# 4.2b: Read 10 notes from the followup_notes.csv and run through the AI model
# to come up with the insights on them

notes_df = pd.read_csv('/content/followup_notes.csv')

# Get the frist 10 rows
notes_to_analyze_df = notes_df.head(10)


insights= []
calls_made_for_4_2 = 0

for _,row in notes_to_analyze_df.iterrows():

  insight = {}
  insight['note_id'] = row['note_id']
  insight['appointment_id'] = row['appointment_id']

  if pd.isna(row['note_text']) or str(row['note_text']).strip() == "":

    insight['root_cause'] = 'other'
    insight['confidence'] = 0.0
    insight['summary'] = 'No note recorded'
    insight['recommended_action'] = 'Re-call patient'
    insight['validation_status'] = 'skipped_blank'

  else:
    output, calls_made_for_4_2 = analyse_note(row['note_text'],calls_made_for_4_2)

    if validateResp(output):
      insight['root_cause'] = output['root_cause']
      insight['confidence'] = output['confidence']
      insight['summary'] = output['summary']
      insight['recommended_action'] = output['recommended_action']
      insight['validation_status'] = 'valid'
      insight['raw_response'] = output
    else:
      insight['validation_status'] = 'invalid'

  insights.append(insight)
  # Sleep for some time before making the next call to respect the free-tier rate limit.
  time.sleep(13)



Validation Successfull
Insight is: root_cause='forgot_or_no_reminder' confidence=0.98 summary='Patient missed the appointment because she received no reminder message and forgot. She requested to reschedule for next week.' recommended_action="Rebook the patient's appointment for next week and ensure reminder messages are properly set up."
Type of object: <class '__main__.FollowupInsight'>
Validation Successfull
Insight is: root_cause='distance_or_transport' confidence=1.0 summary='Patient missed the routine check-up because they live far away and could not find transportation that morning.' recommended_action='Discuss alternative transportation options, telehealth visits, or referring to a closer clinic facility for routine care.'
Type of object: <class '__main__.FollowupInsight'>
Validation Successfull
Insight is: root_cause='cost_or_fee' confidence=0.95 summary='Patient skipped the consultation due to high fees and inquired about follow-up discounts.' recommended_action='Provide info

In [ ]:
import csv

filename = 'notes_analysed.csv'

# Get the column headers
fieldnames = list(insights[0].keys())

with open(filename, mode="w", newline="", encoding="UTF-8") as file:

  writer = csv.DictWriter(file, fieldnames=fieldnames)

  writer.writeheader()

  writer.writerows(insights)

print(f'Successfully wrote data to {filename}')

Successfully wrote data to notes_analysed.csv


## Task 4.3 — Break it on purpose

(a) Feed the Pydantic model two deliberately bad replies and catch each with try/except ValidationError (from pydantic import ValidationError): one with confidence = 1.7, one missing recommended_action; print what each error says.

(b) For notes N007 (Hinglish) and N009 (ambiguous), compare two prompts at temperature=0: V1 = just "Classify this note", V2 = your CRAFT prompt with the allowed values and the instruction to answer other rather than guess. Use the same schema and settings for both (root_cause as a plain string, JSON mode, temperature=0), so that the only difference is the prompt; then check each reply against the allowed list. Show the four results in a small table and say which prompt you would deploy.

(c) Count and print the total number of API calls you used across Tasks 4.1–4.3 (about 14 on a first pass: 1 in 4.1, 9 in 4.2, 4 here; 12 if you reuse your 4.2 results for the V2 prompt).

In [ ]:
# 4.3a: Bad data validations using Pydantic Model
try:
  bad_data1 = FollowupInsight(
      root_cause='Dummy data',
      confidence=1.7,
      summary='Dummy data',
      recommended_action='Dummy data'
    )

except ValidationError as ve:

    print("Data validation failed with the error:")
    print(ve)

# validateResp(bad_data1)

print('\n')

try:
  bad_data2 = FollowupInsight(root_cause='Dummy data',confidence=0.7,summary='Dummy data')

except ValidationError as ve:

    print("Data validation failed with the error:")
    print(ve)


Data validation failed with the error:
1 validation error for FollowupInsight
confidence
  Input should be less than or equal to 1 [type=less_than_equal, input_value=1.7, input_type=float]
    For further information visit https://errors.pydantic.dev/2.14/v/less_than_equal


Data validation failed with the error:
1 validation error for FollowupInsight
recommended_action
  Field required [type=missing, input_value={'root_cause': 'Dummy dat...'summary': 'Dummy data'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.14/v/missing


In [ ]:
import os
import json
import pandas as pd
from google import genai
from google.genai import types


# 4.3b: Compare a non-explanatory prompt to the precisly describing required action prompt

SCHEMA = {   # same idea as the name / age / city example in class: properties, their types, and required
    "type": "object",
    "properties": {"root_cause": {"type": "string"}, "confidence": {"type": "number"},
                   "summary": {"type": "string"}, "recommended_action": {"type": "string"}},
    "required": ["root_cause", "confidence", "summary", "recommended_action"]}

API_KEY = input("Paste your key: ")

client = genai.Client(api_key=API_KEY)

MODEL_NAME = "gemini-3.5-flash"

def analyse_note_v1(note_text,call_count=0):
  prompt = f"""
  Extract the root cause, confidence, summary, and recommended action from this note.

  Followup Note:
  {note_text}
  """

  # Define model behavior
  SYSTEM_TEXT="""
  You are a clinic reception assitant.

  Extract information accurately from patient followup notes.

  Rules:
  - Do not invent information.
  - Only use information present in the notes.
  - Preserve names accurately.
  - Extract every explicit action item.
  - Return only the requested structured information.
  """

  response = client.models.generate_content(
      model=MODEL_NAME,                      # the Flash model from Session 22.1
      contents=prompt,
      config=types.GenerateContentConfig(system_instruction=SYSTEM_TEXT, temperature=0,
          max_output_tokens=1000, response_mime_type="application/json", response_schema=SCHEMA,
          thinking_config=types.ThinkingConfig(thinking_level="minimal")))   # as in Session 22.1; if your model rejects it, delete this line

  call_count = call_count + 1
  data = json.loads(response.text)

  return data, call_count

Paste your key: 


In [ ]:
import pandas as pd


# Read the Note_text of N007 and N009
notes_df = pd.read_csv('/content/followup_notes.csv')

# Retrieve the note_text for the required followup notes
N007_note_text = notes_df[notes_df['note_id'] == 'N007']['note_text'].iloc[0]
N009_note_text = notes_df[notes_df['note_id'] == 'N009']['note_text'].iloc[0]


calls_made_for_4_3 = 0

# Call the unpolished prompted model
N007_resp, calls_made_for_4_3 = analyse_note_v1(N007_note_text,calls_made_for_4_3)
N009_resp, calls_made_for_4_3 = analyse_note_v1(N009_note_text,calls_made_for_4_3)

print(f"N007 Note: {N007_note_text}      Analysis: {N007_resp}")
print(f"N009 Note: {N009_note_text}      Analysis: {N009_resp}")


N007 Note: Bhool gaya, SMS nahi aaya tha. Dobara booking karni hai.      Analysis: {'root_cause': 'Did not receive SMS notification, forgot appointment', 'confidence': 0.95, 'summary': 'Patient forgot the appointment because they did not receive an SMS reminder and needs to rebook.', 'recommended_action': 'Rebook the appointment for the patient'}
N009 Note: Phone was switched off, could not reach the patient after three attempts.      Analysis: {'root_cause': 'Phone was switched off, could not reach the patient after three attempts.', 'confidence': 1.0, 'summary': 'Unable to contact patient as their phone was powered off despite three attempts.', 'recommended_action': 'Attempt to contact the patient at a later time or via an alternative contact method.'}
2


In [ ]:
from tabulate import tabulate

# Read the detail prompted response from the saved csv file
analysed_notes_df = pd.read_csv('/content/notes_analysed.csv')
N007_analyzed_V2_resp = analysed_notes_df[analysed_notes_df['note_id'] == 'N007']['raw_response'].iloc[0]
N009_analyzed_V2_resp = analysed_notes_df[analysed_notes_df['note_id'] == 'N009']['raw_response'].iloc[0]

to_print = [
    {
    "note_id" : "N007",
    "V1 Response" : N007_resp,
    "v2 Response" : N007_analyzed_V2_resp
    },
            {
                "note_id" : "N009",
    "V1 Response" : N009_resp,
    "v2 Response" : N009_analyzed_V2_resp
            }
    ]

print(tabulate(to_print,headers="keys",tablefmt="grid"),)

# Conclusion:
# Looking at the responses, the response elements, especially the 'root cause'
# is more easy to interpret and understand in the CRAFT based prompt (V2). So
# the preferred prompt is CRAFT version


+-----------+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
| note_id   | V1 Response                                                                                                                                                                                                                                                                                                                                                     

In [ ]:

# 4.3c: The total number of API calls you used across Tasks 4.1–4.3

print(f"Call made to Gemini Flash for 4.1: {calls_made_for_4_1}")
print(f"Call made to Gemini Flash for 4.2: {calls_made_for_4_2}")
print(f"Call made to Gemini Flash for 4.3: {calls_made_for_4_3}")

## Task 4.4 — Recommendation and GitHub

(a) Merge notes_analysed.csv with appointments_clean.csv on appointment_id and cross-tabulate root_cause against reminder_sent (pd.crosstab).

Write README.md with:
- a 4-line summary (one finding per Part, each with its number);
- a one-paragraph recommendation that answers the business question using at least four numbers from Parts 1–4;
- the model you chose and why (include AUC and the threshold);
- and three limitations (choose from: the 10 notes are an illustration and cannot show the real reasons clinic-wide; the cleaning statistics came from all rows before the split; the same patient can sit in both train and test; the split is random, not past-to-future; one train/test split; a threshold that really depends on call capacity; clinic-level sample sizes; the data are observational, so associations are not causes).

(b) Create the public GitHub repository, upload every file listed under Submission, confirm the key is nowhere in it, and paste the link at the top of the README.

In [ ]:
import pandas as pd

# 4.4a.1: Merge the 2 CSV files and do a crosstab on the root_cause and reminder sent
appts_df = pd.read_csv('/content/appointments_clean.csv')
analysed_notes_df = pd.read_csv('/content/notes_analysed.csv')

merged_df = pd.merge(appts_df, analysed_notes_df, on='appointment_id',how='inner')

# Basic Frequency Crosstab
basic_crosstab = pd.crosstab(merged_df['root_cause'],merged_df['reminder_sent'])
print("--- Basic Frequency Crosstab ---")
print(basic_crosstab)

# Crosstab with Margins (Adds Row and Column totals)
margin_crosstab = pd.crosstab(merged_df['root_cause'],merged_df['reminder_sent'], margins=True)
print("\n--- Crosstab with Totals (Margins) ---")
print(margin_crosstab)

# 4. Normalized Crosstab
norm_crosstab = pd.crosstab(merged_df['root_cause'],merged_df['reminder_sent'], normalize='index')
print("\n--- Row-Normalized Crosstab (Proportions) ---")
print(norm_crosstab)


--- Basic Frequency Crosstab ---
reminder_sent          No  Yes
root_cause                    
cost_or_fee             0    1
distance_or_transport   1    1
forgot_or_no_reminder   2    0
illness_or_emergency    0    1
other                   1    1
work_or_schedule        1    1

--- Crosstab with Totals (Margins) ---
reminder_sent          No  Yes  All
root_cause                         
cost_or_fee             0    1    1
distance_or_transport   1    1    2
forgot_or_no_reminder   2    0    2
illness_or_emergency    0    1    1
other                   1    1    2
work_or_schedule        1    1    2
All                     5    5   10

--- Row-Normalized Crosstab (Proportions) ---
reminder_sent           No  Yes
root_cause                     
cost_or_fee            0.0  1.0
distance_or_transport  0.5  0.5
forgot_or_no_reminder  1.0  0.0
illness_or_emergency   0.0  1.0
other                  0.5  0.5
work_or_schedule       0.5  0.5
